<a href="https://colab.research.google.com/github/ancestor9/mathematics-for-machine-learning/blob/main/pytorch/03_autograd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [ ]:
import torch

x = torch.randn(3, requires_grad=True)
y = x + 2

# y was created as a result of an operation, so it has a grad_fn attribute.
# grad_fn: references a Function that has created the Tensor
print(x) # created by the user -> grad_fn is None
print(y)
print(y.grad_fn)

tensor([-0.9187,  0.6927, -0.9452], requires_grad=True)
tensor([1.0813, 2.6927, 1.0548], grad_fn=<AddBackward0>)


In [ ]:
# Do more operations on y
z = y * y * 3
print(z)

tensor([ 3.5078, 21.7517,  3.3376], grad_fn=<MulBackward0>)


In [ ]:
z = z.mean()
print(z)

tensor(9.5323, grad_fn=<MeanBackward0>)


In [ ]:
print(x.grad) # dz/dx

None


In [ ]:
# Let's compute the gradients with backpropagation
# When we finish our computation we can call .backward() and have all the gradients computed automatically.
# The gradient for this tensor will be accumulated into .grad attribute.
# It is the partial derivate of the function w.r.t. the tensor

z.backward()

In [ ]:
print(x.grad) # dz/dx

tensor([2.1626, 5.3854, 2.1095])


### 🔗 Chain Rule with `z.mean()` (PyTorch Autograd)

---

#### 1. Computation Graph

$$
x \rightarrow y = x + 2 \rightarrow z = 3y^2
$$

---

#### 2. Chain Rule (Before `mean()`)

$$
\frac{dz}{dx} = \frac{dz}{dy} \cdot \frac{dy}{dx}
$$

$$
\frac{dz}{dy} = 6y,\quad \frac{dy}{dx} = 1
$$

$$
\Rightarrow \frac{dz}{dx} = 6y = 6(x+2)
$$

---

#### 3. Apply `z.mean()`

이제 실제로 미분하는 대상은:

$$
L = \mathrm{mean}(z) = \frac{1}{n} \sum_{i=1}^{n} z_i
$$

---

#### 4. Chain Rule (After `mean()`)

$$
\frac{dL}{dx} = \frac{dL}{dz} \cdot \frac{dz}{dx}
$$

$$
\frac{dL}{dz} = \frac{1}{n}
$$

---

#### 5. Final Gradient

$$
\frac{dL}{dx} = \frac{1}{n} \cdot 6(x+2)
$$

---

#### 6. Example (n = 3)

$$
\frac{dL}{dx} = 2(x+2)
$$

---

In [ ]:
print(x.grad) # dz/dx
2*(x +2)

tensor([2.1626, 5.3854, 2.1095])


tensor([2.1626, 5.3854, 2.1095], grad_fn=<MulBackward0>)

In [ ]:
print(y.grad) # dz/dx

None


/tmp/ipykernel_33835/2512833117.py:1: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:492.)
  print(y.grad) # dz/dx


PyTorch의 효율적인 메모리 관리 설계 때문인 아주 정상적인 현상입니다. 에러가 아니라 **"중간 과정은 굳이 저장하지 않겠다"** 는 선언

결론은, y는 'Leaf Tensor(잎 텐서)'가 아니기 때문이다.

### Leaf Tensor란 무엇인가?
PyTorch의 연산 그래프(Computational Graph)에서 텐서는 두 종류로 나뉩니다.

- Leaf Tensor (잎 텐서): 사용자가 직접 생성한 텐서입니다 (예: x = torch.randn(3, requires_grad=True)). 신경망에서는 대개 **가중치(Weights)** 와 **편향(Bias)** 이 여기에 해당

- Non-Leaf Tensor (중간 변수): 다른 텐서로부터 연산되어 만들어진 결과물 (예: y = x + 2).

In [ ]:
# Generally speaking, torch.autograd is an engine for computing vector-Jacobian product
# It computes partial derivates while applying the chain rule

# -------------
# Model with non-scalar output:
# If a Tensor is non-scalar (more than 1 elements), we need to specify arguments for backward()
# specify a gradient argument that is a tensor of matching shape.
# needed for vector-Jacobian product

x = torch.randn(3, requires_grad=True)
x

y = x * 2
y
print('*'*100)

for _ in range(10):
    y = y * 2
    print(y)

print(y)
print(y.shape)


tensor([1.0521, 0.7717, 0.7341], requires_grad=True)

tensor([2.1041, 1.5433, 1.4681], grad_fn=<MulBackward0>)

****************************************************************************************************
tensor([4.2083, 3.0867, 2.9362], grad_fn=<MulBackward0>)
tensor([8.4166, 6.1734, 5.8725], grad_fn=<MulBackward0>)
tensor([16.8332, 12.3467, 11.7450], grad_fn=<MulBackward0>)
tensor([33.6664, 24.6934, 23.4899], grad_fn=<MulBackward0>)
tensor([67.3328, 49.3868, 46.9798], grad_fn=<MulBackward0>)
tensor([134.6656,  98.7736,  93.9597], grad_fn=<MulBackward0>)
tensor([269.3311, 197.5472, 187.9194], grad_fn=<MulBackward0>)
tensor([538.6623, 395.0944, 375.8387], grad_fn=<MulBackward0>)
tensor([1077.3246,  790.1888,  751.6774], grad_fn=<MulBackward0>)
tensor([2154.6492, 1580.3777, 1503.3549], grad_fn=<MulBackward0>)
tensor([2154.6492, 1580.3777, 1503.3549], grad_fn=<MulBackward0>)
torch.Size([3])


In [ ]:
v = torch.tensor([0.1, 1.0, 0.0001], dtype=torch.float32)
y.backward(v)
print(x.grad)

tensor([2.0480e+02, 2.0480e+03, 2.0480e-01])


In [ ]:

# -------------
# Stop a tensor from tracking history:
# For example during our training loop when we want to update our weights
# then this update operation should not be part of the gradient computation
# - x.requires_grad_(False)
# - x.detach()
# - wrap in 'with torch.no_grad():'

# .requires_grad_(...) changes an existing flag in-place.
a = torch.randn(2, 2)
print(a.requires_grad)
b = ((a * 3) / (a - 1))
print(b.grad_fn)
a.requires_grad_(True)
print(a.requires_grad)
b = (a * a).sum()
print(b.grad_fn)


False
None


tensor([[-1.7725, -0.0419],
        [ 1.1099,  0.7762]], requires_grad=True)

True
True
False
True
False
tensor([3., 3., 3., 3.])


tensor([0., 0., 0., 0.])

tensor([3., 3., 3., 3.])


tensor([0., 0., 0., 0.])

tensor([3., 3., 3., 3.])


tensor([0., 0., 0., 0.])

tensor([0.1000, 0.1000, 0.1000, 0.1000], requires_grad=True)
tensor(4.8000, grad_fn=<SumBackward0>)


In [ ]:

# .detach(): get a new Tensor with the same content but no gradient computation:
a = torch.randn(2, 2, requires_grad=True)
print(a.requires_grad)
b = a.detach()
print(b.requires_grad)

# wrap in 'with torch.no_grad():'
a = torch.randn(2, 2, requires_grad=True)
print(a.requires_grad)
with torch.no_grad():
    print((x ** 2).requires_grad)


True
False
True
False


In [ ]:
# -------------
# backward() accumulates the gradient for this tensor into .grad attribute.
# !!! We need to be careful during optimization !!!
# Use .zero_() to empty the gradients before a new optimization step!
weights = torch.ones(4, requires_grad=True)

for epoch in range(3):
    # just a dummy example
    model_output = (weights*3).sum()
    model_output.backward()

    print(weights.grad)

    # optimize model, i.e. adjust weights...
    with torch.no_grad():
        weights -= 0.1 * weights.grad

    # this is important! It affects the final weights & output
    weights.grad.zero_()

print(weights)
print(model_output)


tensor([3., 3., 3., 3.])


tensor([0., 0., 0., 0.])

tensor([3., 3., 3., 3.])


tensor([0., 0., 0., 0.])

tensor([3., 3., 3., 3.])


tensor([0., 0., 0., 0.])

tensor([0.1000, 0.1000, 0.1000, 0.1000], requires_grad=True)
tensor(4.8000, grad_fn=<SumBackward0>)


In [ ]:
weights

tensor([0.1000, 0.1000, 0.1000, 0.1000], requires_grad=True)

In [ ]:

# Optimizer has zero_grad() method
optimizer = torch.optim.SGD([weights], lr=0.1)
# During training:
optimizer.step()
optimizer.zero_grad()

## 1. Optimizer를 쓰지 않는 경우 (Manual Update)

이 방식은 우리가 수학 공식을 직접 코드로 구현하는 방식으로 미분값을 꺼내서(grad), 학습률을 곱하고, 직접 빼줘야 함

In [ ]:
import torch

# 데이터와 가중치 설정
x = torch.tensor([1, 2, 3, 4], dtype=torch.float32)
y = torch.tensor([2, 4, 6, 8], dtype=torch.float32) # 정답: y = 2x
w = torch.tensor(0.0, dtype=torch.float32, requires_grad=True)

learning_rate = 0.01

for epoch in range(10):
    # 1. 예측 및 손실 계산
    y_pred = x * w
    loss = ((y_pred - y)**2).mean()

    # 2. 역전파 (미분 계산)
    loss.backward()

    # 3. 가중치 업데이트 (수동)
    with torch.no_grad():
        # w = w - lr * grad 공식을 직접 구현
        w -= learning_rate * w.grad

    # 4. 미분값 초기화 (수동)
    w.grad.zero_()

    print(f'Epoch {epoch+1}: w = {w.item():.3f}, loss = {loss.item():.3f}')

tensor(0.)

Epoch 1: w = 0.300, loss = 30.000


tensor(0.)

Epoch 2: w = 0.555, loss = 21.675


tensor(0.)

...

Epoch 8: w = 1.455, loss = 3.083


tensor(0.)

Epoch 9: w = 1.537, loss = 2.228


tensor(0.)

Epoch 10: w = 1.606, loss = 1.609


## 2. Optimizer를 사용하는 경우 (With Adam)

실제 딥러닝에서 사용하는 방식으로 업데이트 로직과 초기화 로직을 optimizer 객체에 맡기는 경우로 여기서는 가장 성능이 좋은 Adam

In [ ]:
import torch

# 데이터와 가중치 설정
x = torch.tensor([1, 2, 3, 4], dtype=torch.float32)
y = torch.tensor([2, 4, 6, 8], dtype=torch.float32)
w = torch.tensor(0.0, dtype=torch.float32, requires_grad=True)

# Optimizer 정의 (어떤 가중치를, 어떤 방식으로 학습할지 결정)
learning_rate = 0.01
# 두 번째 코드를 아래처럼 바꾸면 첫 번째와 결과가 거의 같아짐
optimizer = torch.optim.SGD([w], lr=learning_rate)

for epoch in range(10):
    # 1. 예측 및 손실 계산
    y_pred = x * w
    loss = ((y_pred - y)**2).mean()

    # 2. 미분값 초기화 (Optimizer 사용)
    optimizer.zero_grad()

    # 3. 역전파 (미분 계산)
    loss.backward()

    # 4. 가중치 업데이트 (Optimizer 사용)
    optimizer.step()

    print(f'Epoch {epoch+1}: w = {w.item():.3f}, loss = {loss.item():.3f}')

Epoch 1: w = 0.300, loss = 30.000
Epoch 2: w = 0.555, loss = 21.675
Epoch 3: w = 0.772, loss = 15.660
Epoch 4: w = 0.956, loss = 11.314
Epoch 5: w = 1.113, loss = 8.175
Epoch 6: w = 1.246, loss = 5.906
Epoch 7: w = 1.359, loss = 4.267
Epoch 8: w = 1.455, loss = 3.083
Epoch 9: w = 1.537, loss = 2.228
Epoch 10: w = 1.606, loss = 1.609
